# Cross-hospital check — Romanian external split

The Romanian dataset (Zenodo 15487430) ships a 180-image external-validation
split from clinics that are not in its training set. This notebook measures
how the shipped models behave on that hospital. Two caveats matter:

- **Disease GT.** Romanian `CAR`/`API` boxes are lesion-level (~0.1% of the
  image); DENTEX labels the whole affected tooth (~1.2%). They are converted
  to tooth-level with the production tooth detector (the same conversion used
  for v4 training), so the GT inherits tooth-detector errors.
- **Findings GT.** Romanian classes are mapped to the nearest DentalOPG class
  (`OBT`→Filling, `END`→Root Canal, `PRR`→Cap, `BON`→Periodontitis,
  `API`→Apical Abscess, `IMT`→Horizontal Impaction). The box definitions and
  class meanings differ, so the findings score is a lower bound, not a
  like-for-like comparison.

The point is not a headline metric; it is to show that neither model can be
assumed to transfer to a new hospital without re-annotation.


In [ ]:
import json
import os
import shutil
from collections import Counter
from pathlib import Path

from ultralytics import YOLO

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROD = REPO / "dentex_week3" / "production"
DATA = Path("/home/sayfeldin/Projects/dental-model-research")
ROM = DATA / "data/raw/romania/ds/test_alte_cabinete/Ext-validation"
OUT = Path("/tmp/opencode/cross_hospital")
shutil.rmtree(OUT, ignore_errors=True)
assert ROM.exists(), f"download the Romanian dataset first: {ROM}"

ROM_NAMES = ["IMP", "PRR", "OBT", "END", "CAR", "BON", "IMT", "API",
             "ROT", "FUR", "APS", "ROR", "ORD", "SRD"]
DISEASE_NAMES = ["Impacted", "Caries", "Periapical Lesion"]
FINDING_NAMES = ["BDR", "Bridge", "Cap", "Distoangular Impaction", "Filling",
                 "Horizontal Impaction", "Missing Teeth", "Periodontitis",
                 "Root Canal", "Unerupted Wisdom", "Apical Abscess"]

tooth = YOLO(str(PROD / "tooth_best.pt"))
disease = YOLO(str(PROD / "disease_best.pt"))
findings = YOLO(str(PROD / "findings_best.pt"))
print("models loaded")


In [ ]:
# ---- disease: convert CAR/API lesions to tooth-level GT ----
D_MAP = {"CAR": 1, "API": 2, "IMT": 0}
d_root = OUT / "disease"
for sub in ("images", "labels"):
    (d_root / sub).mkdir(parents=True, exist_ok=True)

kept, skipped = 0, 0
d_boxes = Counter()
for img in sorted((ROM / "images").glob("*.jpg")):
    lab = ROM / "labels" / (img.stem + ".txt")
    if not lab.exists():
        continue
    rows = [l.split() for l in lab.read_text().splitlines() if l.strip()]
    lesions = [(ROM_NAMES[int(r[0])], [float(v) for v in r[1:]]) for r in rows
               if ROM_NAMES[int(r[0])] in D_MAP]
    needs_teeth = any(n in ("CAR", "API") for n, _ in lesions)
    if needs_teeth:
        res = tooth.predict(str(img), imgsz=1024, conf=0.25, verbose=False)[0]
        W, H = res.orig_shape[1], res.orig_shape[0]
        tboxes = res.boxes.xyxy.tolist()
    else:
        tboxes, W, H = [], None, None

    out, converted_target = [], 0
    for name, (cx, cy, w, h) in lesions:
        cls = D_MAP[name]
        if name == "IMT":
            box = (cx, cy, w, h)
        else:
            px, py = cx * W, cy * H
            containing = [tb for tb in tboxes
                          if tb[0] <= px <= tb[2] and tb[1] <= py <= tb[3]
                          and 0.002 <= ((tb[2] - tb[0]) * (tb[3] - tb[1])) / (W * H) <= 0.05]
            if not containing:
                continue
            tb = min(containing, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))
            x1, y1, x2, y2 = tb
            box = ((x1 + x2) / 2 / W, (y1 + y2) / 2 / H,
                   (x2 - x1) / W, (y2 - y1) / H)
            converted_target += 1
        out.append(f"{cls} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}")
        d_boxes[DISEASE_NAMES[cls]] += 1

    if needs_teeth and converted_target == 0:
        skipped += 1
        continue
    os.symlink(img, d_root / "images" / img.name)
    (d_root / "labels" / (img.stem + ".txt")).write_text(
        "\n".join(out) + ("\n" if out else ""))
    kept += 1

(d_root / "test.txt").write_text(
    "\n".join(str(p) for p in sorted((d_root / "images").iterdir())) + "\n")
(d_root / "data.yaml").write_text(json.dumps({
    "train": str(d_root / "test.txt"), "val": str(d_root / "test.txt"),
    "test": str(d_root / "test.txt"),
    "names": {i: n for i, n in enumerate(DISEASE_NAMES)}}, indent=2))
print(f"disease GT: {kept} images, {skipped} skipped, {dict(d_boxes)}")


In [ ]:
# ---- disease model on the converted external GT ----
m = disease.val(data=str(OUT / "disease/data.yaml"), split="test",
                imgsz=1024, conf=0.001, iou=0.6, verbose=False)
per_class = {m.names[int(i)]: float(a)
             for i, a in zip(m.box.ap_class_index, m.box.ap50)}
summary = {
    "split": "Romanian external validation (tooth-converted GT)",
    "images": kept,
    "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
    "precision": float(m.box.mp), "recall": float(m.box.mr),
    "ap50_per_class": per_class,
}
(REPO / "dentex_week3/eval/disease_cross_hospital.json").write_text(
    json.dumps(summary, indent=2))
print(f"mAP50 {summary['mAP50']:.1%}  mAP50-95 {summary['mAP50_95']:.1%}  "
      f"P {summary['precision']:.1%}  R {summary['recall']:.1%}")
for n, a in summary["ap50_per_class"].items():
    print(f"  {n:22s} AP50 {a:.3f}")


In [ ]:
# ---- findings: map Romanian classes to nearest DentalOPG class ----
F_MAP = {2: 4, 3: 8, 1: 2, 5: 7, 7: 10, 6: 5}
f_root = OUT / "findings"
for sub in ("images", "labels"):
    (f_root / sub).mkdir(parents=True, exist_ok=True)

kept_f = 0
f_boxes = Counter()
for img in sorted((ROM / "images").glob("*.jpg")):
    lab = ROM / "labels" / (img.stem + ".txt")
    if not lab.exists():
        continue
    out = []
    for line in lab.read_text().splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        new = F_MAP.get(int(parts[0]))
        if new is None:
            continue
        out.append(f"{new} " + " ".join(parts[1:]))
        f_boxes[FINDING_NAMES[new]] += 1
    if not out:
        continue
    os.symlink(img, f_root / "images" / img.name)
    (f_root / "labels" / (img.stem + ".txt")).write_text("\n".join(out) + "\n")
    kept_f += 1

(f_root / "test.txt").write_text(
    "\n".join(str(p) for p in sorted((f_root / "images").iterdir())) + "\n")
(f_root / "data.yaml").write_text(json.dumps({
    "train": str(f_root / "test.txt"), "val": str(f_root / "test.txt"),
    "test": str(f_root / "test.txt"),
    "names": {i: n for i, n in enumerate(FINDING_NAMES)}}, indent=2))
print(f"findings GT: {kept_f} images, {dict(f_boxes)}")


In [ ]:
# ---- findings model on the mapped external GT ----
m = findings.val(data=str(OUT / "findings/data.yaml"), split="test",
                 imgsz=640, conf=0.001, iou=0.6, verbose=False)
per_class = {m.names[int(i)]: float(a)
             for i, a in zip(m.box.ap_class_index, m.box.ap50)}
summary_f = {
    "split": "Romanian external validation (class-mapped GT)",
    "images": kept_f,
    "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
    "precision": float(m.box.mp), "recall": float(m.box.mr),
    "ap50_per_class": per_class,
}
(REPO / "dentex_week3/eval/findings_cross_hospital.json").write_text(
    json.dumps(summary_f, indent=2))
print(f"mAP50 {summary_f['mAP50']:.1%}  mAP50-95 {summary_f['mAP50_95']:.1%}  "
      f"P {summary_f['precision']:.1%}  R {summary_f['recall']:.1%}")
for n, a in summary_f["ap50_per_class"].items():
    print(f"  {n:24s} AP50 {a:.3f}")


## Reading the result

Both models degrade on a new hospital, but for different reasons:

- The disease model still localises most impacted teeth and a meaningful
  fraction of caries/apical lesions, because those classes share a radiographic
  appearance across hospitals. The tooth-converted GT flatters it slightly,
  since the same tooth detector defines the boxes.
- The findings model collapses when the annotation style changes. Romanian
  restorations/endodontics are tight lesion boxes, while DentalOPG labels the
  tooth; the model has also never seen this hospital's imaging. The zero APs
  for root canal, periodontitis and impaction are partly a mapping artefact,
  but the overall message is real.

Neither model should be used clinically outside the population it was trained
and validated on without re-annotation and a local validation set.
